In [9]:
import requests
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, GenerationConfig, pipeline
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import numpy as np
from vllm import LLM, SamplingParams
from transformers import Pipeline, PreTrainedTokenizer



In [2]:
llm = LLM(
    model="cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0",
    tensor_parallel_size=8,
    max_model_len=4096,
    gpu_memory_utilization=0.5,
)


/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-05-28 10:41:47,941	INFO worker.py:1749 -- Started a local Ray instance.


INFO 05-28 10:41:50 llm_engine.py:100] Initializing an LLM engine (v0.4.2) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
INFO 05-28 10:42:09 utils.py:660] Found nccl from library /home/ubuntu/.config/vllm/nccl/cu12/libnccl.so.2.18.1
(RayWorkerWrapper pid=3353709) INFO 05-28 10:42:09 utils.py:660] Found nccl from library /home/ubuntu/.config/vllm/nccl/cu12/libnccl.so.2.18.1
INFO 05-28 

In [3]:
tokenizer = AutoTokenizer.from_pretrained("cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", trust_remote_cote=True)


In [4]:
messages = [{"role": "user",
             "content": f"""너는 자기소개서 첨삭 전문가야.
주어진 자기소개서를 분석해서 평가해야해.
참고할 사항은 다음과 같아
1. '저는', '제가', '저의'같은 1인칭 주어시첨 표현은 제한적으로 사용.
2. '귀사', '당사', '이 회사' 같은 표현보단 기업의 정식명칭 사용.
3. 문장 구조나 띄어쓰기에 대한 평가는 하지 않음.

출력은 다음의 형식을 따라
[평가]
이런 점이 좋아요:
n. (소제목):
n. (소제목):
...
다시 생각해 봐요:
n. (소제목):
n. (소제목):
...

다음이 자기소개서야 :
[겸손하게 배우는 자세]
평소 디지털 트랜스포메이션에 의해 비즈니스의 속도와 변화가 심해지면서 IT 개발 직무에서도 여러 가지의 역할을 할 수 있어야 한다고 생각했습니다. 따라서 저는 IT와 함께 소셜미디어 매니지먼트를 연계 전공하게 되었습니다. IT와 비즈니스 관련수업을 병행해온 저에게CJ오쇼핑은 일반 IT 회사와 다르게 유통회사의 IT 부서이었던 것이 매력적으로 다가왔습니다. 저는 연계 전공 동안 음식점을 마케팅한 적 있습니다. 음식점에서 단순히 음식을 파는 것이 아니라 고객의 입장에서 생각해보는 것이 중요하다는 것을 깨달았습니다. 또한, 미국을 여행하는 동안 떠오른 여행용 가방과 관련된 아이디어로 창업경진대회까지도 참여했습니다. 입상에는 실패했지만, 도전을 무서워하지 않는 용기를 가지게 되었습니다. 연계 전공을 하며 본 전공에 대해 이해도가 떨어질 수도 있지만, 항상 배우려는 태도로 임하는 겸손한 사람이 되겠습니다. 또한, 최근에는 유럽 16개국을 여행하며 다양한 인종의 사람들을 만나 동행을 하게 되면서 다른 사람의 마음을 잘 파악한다는 이야기를 많이 들었습니다. 평소 남들보다 먼저 행동하고 해결하려 했던 것들이 다른 사람들의 생각을 아는 데 도움이 되었던 것 같습니다. IT 개발 업무에서도 고객의 마음을 빠르게 파악하여 트랜드에 민감하게 반응하는 개발자가 되겠습니다.
제가 바라보는 CJ오쇼핑의 IT기술 강점은 트랜드에 굉장히 민감하게 반응한다는 것입니다. 최근 커머스와 미디어시장의 경제가 무너지고 미디어급변이 일어나면서. CJ오쇼핑은 E&M과 합병을 결정하였습니다. 그에 따라 최근 이슈인 인공지능 음성인식 서비스나 가상현실, 증강현실 등의 개발을 확대해가기에 새로운 경쟁력을 가진다고 생각합니다. 하지만 최근 T커머스에 주력을 다 하면서, 모바일 사업 부분에서 경쟁사들보다 부진하다고 생각합니다. 따라서 다양한 계열사를 가진 장점을 이용해 계열사 간 연계를 강화하고 모바일 최적화 상품과 동영상 콘텐츠를 확대하며 신규 서비스를 적극 추진한다면 고객 확보에 도움이 될 것이라고 생각합니다.
"""}]


In [5]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)


sampling_params = SamplingParams(temperature=0, max_tokens=4096)


outputs = llm.generate(text, sampling_params)


print(outputs[0].outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.28s/it]

 [평가]
이런 점이 좋아요:
1. (다양한 경험):
   - IT와 소셜미디어 매니지먼트를 연계 전공한 경험을 통해 다양한 역할을 수행할 수 있는 능력을 강조한 점이 좋습니다.
   - 음식점 마케팅 경험과 창업경진대회 참여를 통해 도전 정신과 문제 해결 능력을 보여준 점이 긍정적입니다.
   - 유럽 16개국 여행 경험을 통해 다양한 인종의 사람들을 만나면서 다른 사람의 마음을 잘 파악하는 능력을 강조한 점이 좋습니다.

2. (기업 이해도):
   - CJ오쇼핑의 IT 부서 특성에 대해 이해하고 있으며, 이를 바탕으로 자신의 경험을 연결시킨 점이 좋습니다.
   - CJ오쇼핑의 최근 합병과 기술 강점을 언급하며, 회사의 현재 상황에 대한 깊은 이해를 보여준 점이 긍정적입니다.

다시 생각해 봐요:
1. (1인칭 주어 사용):
   - "저는", "제가", "저의" 같은 1인칭 주어 사용이 빈번합니다. 이를 줄이고 더 객관적인 표현을 사용하면 좋겠습니다.

2. (기업 명칭 사용):
   - "CJ오쇼핑"이라는 정식 명칭을 사용한 점은 좋지만, "귀사", "당사" 같은 표현을 사용하지 않도록 주의해야 합니다.

3. (구체적인 기여 방안):
   - CJ오쇼핑의 모바일 사업 부진에 대한 분석과 개선 방안에 대한 구체적인 제안이 부족합니다. 더 구체적인 계획과 실행 방안을 제시하면 좋겠습니다.

4. (겸손한 자세 강조):
   - 겸손한 자세를 강조하는 부분이 좋지만, 이를 통해 어떻게 구체적으로 기여할 수 있는지에 대한 설명이 부족합니다. 자신의 강점과 이를 통해 어떻게 기여할 수 있는지에 대한 구체적인 설명이 필요합니다.


In [2]:
from langchain_community.llms import VLLM

llm = VLLM(
    model="cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0",
    tensor_parallel_size=8,
    max_model_len=4096,
    gpu_memory_utilization=0.5,
)

print(llm("What is the capital of France?"))

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-05-28 11:21:35,200	INFO worker.py:1749 -- Started a local Ray instance.


INFO 05-28 11:21:36 llm_engine.py:100] Initializing an LLM engine (v0.4.2) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
INFO 05-28 11:21:54 utils.py:660] Found nccl from library /home/ubuntu/.config/vllm/nccl/cu12/libnccl.so.2.18.1
(RayWorkerWrapper pid=3393395) INFO 05-28 11:21:54 utils.py:660] Found nccl from library /home/ubuntu/.config/vllm/nccl/cu12/libnccl.so.2.18.1
(RayWorkerW

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(
Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


제가 기본적으로 알고있는 사항이나 상식 같은 것에 대해서 쉽게 물어볼 수 있는 질문들 몇가지를 적어 보겠습니다. 나중에 새로운 사실이나 상식을 추가하면 더 추가할 예정입니다.
* 한국의 위치는 어디인가?
* 한국의 지도는 어떻게 그려지는가?
* 안내양의 도시는 어디인가?
* 인간의 DNA는 무엇으로 이루어져 있는가?


In [1]:
# 캐시 지우기
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel,TextStreamer, pipeline 
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA, LLMChain
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
import torch
import pandas as pd

def generate_talent_description(company_name, retriever, llm):
    # 질의 생성
    query = f""" 
    "회사의 모든 인재상을 입력하세요."
    "인재상 설명 이외의 다른 단어는 출력하지 마세요."
    "질문에 최선을 다해 답변하세요. "
    "검색 가능한 모든 도구를 자유롭게 사용하세요 "
    "필요한 경우에만 관련 정보 제공을 하세요"
    "답을 모르면 모른다고 말하세요. 답을 지어내려고 하지 마세요."
    "반드시 한국어로 답변하세요"
    """
    
    # 답변 생성
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    
    result = qa_chain({"query": query})
    talents = result["result"]


    return result

In [12]:
# 검색 관련 함수
def extract_company_info(company_name):
    # 크롬 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # WebDriver 설정
    driver = webdriver.Chrome(options=chrome_options)

    try:
        # 변수 설정
        base_url = "https://www.jobkorea.co.kr/starter/companyreport"

        # URL 구성 및 접속
        url = f"{base_url}?schTxt={company_name}"
        driver.get(url)

        # 원하는 요소가 로드될 때까지 기다리고 요소 찾기
        element = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.XPATH, f'//dt/a/strong[contains(text(), "기업심층분석 1. {company_name}, 채용분석 및 기업정보")]'))
        )
    
        href = element.find_element(By.XPATH, '..').get_attribute('href')
        print(f"Found detailed analysis link: {href}")

        # 링크로 이동
        driver.get(href)

        # 페이지 소스 가져오기
        page_source = driver.page_source

        # BeautifulSoup으로 페이지 소스 파싱
        soup = BeautifulSoup(page_source, 'html.parser')

        # 'viewWrap' 클래스를 가진 모든 요소 찾기
        view_wrap_elements = soup.find_all(class_="viewWrap")

        # 각 요소에서 텍스트 추출
        extracted_text = ""
        for element in view_wrap_elements:
            text = element.get_text(strip=True)
            extracted_text += text + "\n"

        return extracted_text

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

    finally:
        driver.quit()

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": "cuda"})

In [14]:

# 변수 설정
company_name = "카카오"
report = """
안녕하세요. 저는 호남대학교 컴퓨터공학과 4학년에 재학 중인 이진수입니다. 저는 어려서부터 컴퓨터와 프로그래밍에 관심이 많았고, 대학에 입학한 후에는 소프트웨어 개발에 필요한 다양한 기술과 지식을 배우며 역량을 키워 왔습니다.
저는 학부 과정에서 프로그래밍 언어, 자료구조, 알고리즘, 데이터베이스, 웹 개발 등 컴퓨터공학의 기본 지식을 충실히 학습했습니다. 특히 Java와 Python을 활용한 프로젝트를 통해 실무에 적용 가능한 개발 기술을 연마했습니다. 3학년 때는 '학생 관리 시스템' 프로젝트에 참여하여 웹 애플리케이션 개발 전 과정을 경험하였고, 백엔드 개발을 담당하며 REST API 설계와 데이터베이스 관리 능력을 키웠습니다.
또한, 저는 꾸준한 자기개발을 통해 최신 기술 동향을 파악하고 역량을 강화하고 있습니다. 최근에는 클라우드 컴퓨팅과 머신러닝에 관심을 갖고 AWS와 TensorFlow를 학습하고 있습니다. 이러한 새로운 기술을 프로젝트에 적극적으로 활용하여 혁신적인 솔루션을 개발하는 것이 저의 목표입니다.
"""
# CSV 파일 경로
csv_file_path = "./company_talents.csv"

# CSV 파일 읽기
df = pd.read_csv(csv_file_path)

# # 파이프라인 객체 생성
# hf_pipeline = pipeline("text-generation", model=model_orion, tokenizer=tokenizer_orion, streamer=streamer)
    
# # HuggingFacePipeline으로 모델 래핑
# llm = HuggingFacePipeline(pipeline=hf_pipeline)

# company_name 열에 company_name이 있는지 확인
if company_name in df["company_name"].values:
    # company_name이 존재하는 경우, talents 열의 값을 가져옴
    talents = df[df["company_name"] == company_name]["talents"].tolist()
    # 결과 출력
    print("Talents:", talents)
else:
    # company_name이 존재하지 않는 경우, extract_company_info 함수 실행
    company_info = extract_company_info(company_name)
    if company_info:
        # 문서 임베딩
        vector_store = FAISS.from_texts(texts=[company_info], embedding=embeddings)

        # Retriever 생성
        retriever = vector_store.as_retriever(search_kwargs={"k": 1})

        # 인재상 생성
        talents = generate_talent_description(company_name, retriever, llm)
        print(talents)
        str(talents)

        # 결과를 데이터프레임에 추가
        new_data = {"company_name": company_name, "talents": talents}
        df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)
        
        # 결과를 CSV 파일에 저장
        df.to_csv(csv_file_path, index=False)
        print("인재상이 생성되어 CSV 파일에 추가되었습니다.")
    else:
        print("기업 정보 추출에 실패했습니다. 직접 인재상을 작성해주세요")
        # 인재상 입력
        talents = input("인재상 : ")
        
        # 결과를 데이터프레임에 추가
        new_data = {"company_name": company_name, "talents": talents}
        df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)

        # 결과를 CSV 파일에 저장
        df.to_csv(csv_file_path, index=False)
        print("인재상이 생성되어 CSV 파일에 추가되었습니다.")

Found detailed analysis link: https://www.jobkorea.co.kr/starter/companyreport/view?Inside_No=12806&schCtgr=101012&schGrpCtgr=101&schTxt=%EC%B9%B4%EC%B9%B4%EC%98%A4&Page=1


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]

{'query': ' \n    "회사의 모든 인재상을 입력하세요."\n    "인재상 설명 이외의 다른 단어는 출력하지 마세요."\n    "질문에 최선을 다해 답변하세요. "\n    "검색 가능한 모든 도구를 자유롭게 사용하세요 "\n    "필요한 경우에만 관련 정보 제공을 하세요"\n    "답을 모르면 모른다고 말하세요. 답을 지어내려고 하지 마세요."\n    "반드시 한국어로 답변하세요"\n    ', 'result': ' "카카오스러움", "Willing to Venture", "Back to Basic", "Trust to Trust", "Act for Yourself", "Tech for Good".\n    신뢰도: 85%', 'source_documents': [Document(page_content='카카오는 상시 영입을 기본으로 하며, 인턴의 경우 비정기적으로 채용을 진행한다. 모든 영입은 카카오 인재 영입 사이트에서 공지하고 지원을 받는다. 언제 어떤 직무의 공고가 나올지 예측이 어렵기 때문에 카카오 채용에 관심이 있다면 인재풀에 지원서를 등록해 두는 것도 좋다. 직원 유형은 ▲정규직 ▲계약직 ▲인턴 ▲어시스턴트로 구분한다. 어시스턴트는 아르바이트와 인턴을 혼합한 영입 형태다. 정규/계약직의 경우 오프라인 인터뷰 불합격 시 1년간 재지원할 수 없다(서류 전형 불합격 시 재지원 가능). 카카오 모집 분야는 ▲테크 ▲서비스비즈 ▲디자인/브랜드 ▲스태프로 나뉜다.카카오, 기업 개요카카오는 국내 1위 메신저인 카카오톡을 포함한 다양한 모바일 서비스를 제공 중인 ‘모바일 라이프 플랫폼’ 기업이다. 인터넷 포털 사이트 다음(Daum)을 비롯해 모바일/인터넷 기반의 커머스, 모빌리티, 금융, 게임, 음악, 스토리 IP를 주축으로 사업을 전개하고 있다. 카카오의 사업들은 일상의 영역을 모바일로 연결하면서 새로운 편익을 전달하고 있으며, 카카오톡을 중심으로 시너지를 발산하고 있다. 카카오는 이동의 혁신을 가져오고 있는 카카오모빌리티, 금융 습관의 